In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
n_clusters = 8
vectorizer_model = CountVectorizer(stop_words='english')
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
cluster_model = KMeans(n_clusters=n_clusters, random_state=42)
umap_model = UMAP(n_neighbors=15, n_components=8, min_dist=0.0, metric='cosine', random_state=42)
topic_model = BERTopic(vectorizer_model=vectorizer_model, ctfidf_model=ctfidf_model, hdbscan_model=cluster_model, umap_model=umap_model)

In [3]:
filenames = ['android.parquet', 'r.parquet', 'python.parquet', 'web.parquet', 'csharp.parquet', 'java.parquet', 'other.parquet']
names = ['android', 'r', 'python', 'web', 'csharp', 'java', 'other']

In [4]:
topics_dir = Path('figures/topics/')
topics_dir.mkdir(parents=True, exist_ok=True)

for i, (name, filename) in enumerate(zip(names, filenames)):
    print('*************************************')
    print(f'Processing: {name}')

    df = pd.read_parquet(f'data/parquet/{filename}')
    df.info()

    df = df.dropna(subset=['text'])
    df = df[df['T'] == 1]
    df['embedding'] = df['embedding'].apply(lambda x: np.array(x, dtype=np.float32))
    df_before = df[df['P'] == 0]
    df_after = df[df['P'] == 1]

    # topics before
    print('Topics before:----------------------')
    topics, probs = topic_model.fit_transform(df_before['text'], np.stack(df_before['embedding'].values))
    print(topic_model.get_topic_info())

    fig = topic_model.visualize_barchart(top_n_topics=n_clusters, n_words=10)
    fig.update_layout(font=dict(size=10))
    fig.show()
    fn = f'figures/topics/{name}_topics_before.pdf'
    fig.write_image(fn)

    umap_embeddings = topic_model.umap_model.transform(np.stack(df_before['embedding'].values))
    print('SC: ', silhouette_score(umap_embeddings, topics))

    # topics after
    print('Topics after:----------------------')
    topics, probs = topic_model.fit_transform(df_after['text'], np.stack(df_after['embedding'].values))
    print(topic_model.get_topic_info())
    
    fig = topic_model.visualize_barchart(top_n_topics=n_clusters, n_words=10)
    fig.update_layout(font=dict(size=10))
    fig.show()
    fn = f'figures/topics/{name}_topics_after.pdf'
    fig.write_image(fn)

    umap_embeddings = topic_model.umap_model.transform(np.stack(df_after['embedding'].values))
    print('SC: ', silhouette_score(umap_embeddings, topics))

*************************************
Processing: android
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75353 entries, 0 to 75352
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PostTypeId        75353 non-null  int64  
 1   AcceptedAnswerId  24193 non-null  float64
 2   CreationDate      75353 non-null  object 
 3   Score             75353 non-null  int64  
 4   ViewCount         75353 non-null  float64
 5   AnswerCount       75353 non-null  float64
 6   CommentCount      75353 non-null  int64  
 7   FavoriteCount     6067 non-null   float64
 8   Title             75353 non-null  object 
 9   Tags              75353 non-null  object 
 10  line_count        75353 non-null  float64
 11  T                 75353 non-null  int64  
 12  P                 75353 non-null  int64  
 13  W                 75353 non-null  int64  
 14  code              55913 non-null  object 
 15  text              75278 non-n

  File "c:\Users\denis\AppData\Local\Programs\Python\Python312\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\denis\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\denis\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\denis\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


   Topic  Count                                 Name  \
0      0   3277            0_app_android_device_user   
1      1   2627     1_fragment_code_adapter_activity   
2      2   2542            2_android_want_studio_app   
3      3   2418          3_error_failed_build_gradle   
4      4   2100  4_composable_compose_viewmodel_code   
5      5   2070                 5_code_like_using_im   
6      6   1988         6_android_version_file_error   
7      7   1093           7_widget_flutter_code_data   

                                      Representation  \
0  [app, android, device, user, google, play, usi...   
1  [fragment, code, adapter, activity, recyclervi...   
2  [android, want, studio, app, image, flutter, l...   
3  [error, failed, build, gradle, project, run, t...   
4  [composable, compose, viewmodel, code, functio...   
5  [code, like, using, im, app, function, activit...   
6  [android, version, file, error, project, studi...   
7  [widget, flutter, code, data, error, page, c

SC:  0.376676
Topics after:----------------------
   Topic  Count                            Name  \
0      0   3173       0_code_function_want_data   
1      1   2200       1_app_android_device_user   
2      2   2184     2_android_want_studio_image   
3      3   1911      3_android_file_app_project   
4      4   1909  4_code_activity_fragment_class   
5      5   1748            5_code_using_user_im   
6      6   1530    6_build_gradle_project_error   
7      7    827  7_error_failed_duplicate_crash   

                                      Representation  \
0  [code, function, want, data, like, class, list...   
1  [app, android, device, user, google, screen, w...   
2  [android, want, studio, image, flutter, app, l...   
3  [android, file, app, project, version, build, ...   
4  [code, activity, fragment, class, adapter, rec...   
5  [code, using, user, im, like, app, want, use, ...   
6  [build, gradle, project, error, run, version, ...   
7  [error, failed, duplicate, crash, excep

SC:  0.33108866
*************************************
Processing: r
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87578 entries, 0 to 87577
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   PostTypeId        87578 non-null  int64  
 1   AcceptedAnswerId  46537 non-null  float64
 2   CreationDate      87578 non-null  object 
 3   Score             87578 non-null  int64  
 4   ViewCount         87578 non-null  float64
 5   AnswerCount       87578 non-null  float64
 6   CommentCount      87578 non-null  int64  
 7   FavoriteCount     6406 non-null   float64
 8   Title             87578 non-null  object 
 9   Tags              87578 non-null  object 
 10  line_count        87578 non-null  float64
 11  T                 87578 non-null  int64  
 12  P                 87578 non-null  int64  
 13  W                 87578 non-null  int64  
 14  code              80223 non-null  object 
 15  text              8

...

SC:  0.2992499
*************************************
Processing: java
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134350 entries, 0 to 134349
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   PostTypeId        134350 non-null  int64  
 1   AcceptedAnswerId  47990 non-null   float64
 2   CreationDate      134350 non-null  object 
 3   Score             134350 non-null  int64  
 4   ViewCount         134350 non-null  float64
 5   AnswerCount       134350 non-null  float64
 6   CommentCount      134350 non-null  int64  
 7   FavoriteCount     10634 non-null   float64
 8   Title             134350 non-null  object 
 9   Tags              134350 non-null  object 
 10  line_count        134350 non-null  float64
 11  T                 134350 non-null  int64  
 12  P                 134350 non-null  int64  
 13  W                 134350 non-null  int64  
 14  code              116809 non-null  object 
 15

SC:  0.35574692
Topics after:----------------------
   Topic  Count                            Name  \
0      0   6678        0_method_class_code_list   
1      1   3985          1_like_method_code_way   
2      2   3948    2_java_using_application_use   
3      3   2921        3_error_spring_code_test   
4      4   2454    4_code_button_activity_image   
5      5   2261  5_error_version_exception_file   
6      6   2013    6_version_project_java_error   
7      7   1342     7_pomxml_project_maven_file   

                                      Representation  \
0  [method, class, code, list, array, object, lik...   
1  [like, method, code, way, want, using, use, st...   
2  [java, using, application, use, way, project, ...   
3  [error, spring, code, test, using, request, cl...   
4  [code, button, activity, image, app, fragment,...   
5  [error, version, exception, file, run, applica...   
6  [version, project, java, error, file, run, bui...   
7  [pomxml, project, maven, file, depend

SC:  0.3537984
*************************************
Processing: other
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1278394 entries, 0 to 1278393
Data columns (total 17 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   PostTypeId        1278394 non-null  int64  
 1   AcceptedAnswerId  491159 non-null   float64
 2   CreationDate      1278394 non-null  object 
 3   Score             1278394 non-null  int64  
 4   ViewCount         1278394 non-null  float64
 5   AnswerCount       1278394 non-null  float64
 6   CommentCount      1278394 non-null  int64  
 7   FavoriteCount     75069 non-null    float64
 8   Title             1278394 non-null  object 
 9   Tags              1278394 non-null  object 
 10  line_count        1278394 non-null  float64
 11  T                 1278394 non-null  int64  
 12  P                 1278394 non-null  int64  
 13  W                 1278394 non-null  int64  
 14  code              989247 no

SC:  0.3426666
Topics after:----------------------
   Topic  Count                            Name  \
0      0  65284            0_using_app_data_way   
1      1  44284            1_code_im_error_view   
2      2  39114           2_using_like_use_file   
3      3  36355      3_function_code_type_class   
4      4  28882       4_table_query_column_date   
5      5  28145     5_file_docker_error_command   
6      6  18408         6_json_array_like_using   
7      7  13817  7_error_terraform_file_version   

                                      Representation  \
0  [using, app, data, way, use, want, need, im, u...   
1  [code, im, error, view, page, using, user, met...   
2  [using, like, use, file, im, tried, want, way,...   
3  [function, code, type, class, im, like, array,...   
4  [table, query, column, date, rows, columns, ro...   
5  [file, docker, error, command, run, script, co...   
6  [json, array, like, using, error, following, d...   
7  [error, terraform, file, version, foll

SC:  0.33025965


In [5]:
df = pd.read_parquet(f'data/parquet/python.parquet')
df.info()

df = df.dropna(subset=['text'])
df = df[df['T'] == 1]
df['embedding'] = df['embedding'].apply(lambda x: np.array(x, dtype=np.float32))
df_before = df[df['P'] == 0]
df_after = df[df['P'] == 1]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459818 entries, 0 to 459817
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   PostTypeId        459818 non-null  int64  
 1   AcceptedAnswerId  197744 non-null  float64
 2   CreationDate      459818 non-null  object 
 3   Score             459818 non-null  int64  
 4   ViewCount         459818 non-null  float64
 5   AnswerCount       459818 non-null  float64
 6   CommentCount      459818 non-null  int64  
 7   FavoriteCount     27917 non-null   float64
 8   Title             459818 non-null  object 
 9   Tags              459818 non-null  object 
 10  line_count        459818 non-null  float64
 11  T                 459818 non-null  int64  
 12  P                 459818 non-null  int64  
 13  W                 459818 non-null  int64  
 14  code              414430 non-null  object 
 15  text              458859 non-null  object 
 16  embedding         45

In [6]:
# topics before
topics, probs = topic_model.fit_transform(df_before['text'], np.stack(df_before['embedding'].values))
print(topic_model.get_topic_info())

   Topic  Count                            Name  \
0      0  21092     0_list_string_want_function   
1      1  19728  1_dataframe_column_like_values   
2      2  18308         2_python_using_file_use   
3      3  16719         3_error_file_python_run   
4      4  16577   4_class_code_function_program   
5      5  12112         5_plot_model_image_code   
6      6  12097     6_django_viewspy_user_error   
7      7   2212              7_nan_column_10_id   

                                      Representation  \
0  [list, string, want, function, like, code, num...   
1  [dataframe, column, like, values, list, output...   
2  [python, using, file, use, run, im, way, scrip...   
3  [error, file, python, run, using, install, cod...   
4  [class, code, function, program, im, button, m...   
5  [plot, model, image, code, error, data, using,...   
6  [django, viewspy, user, error, model, database...   
7  [nan, column, 10, id, dataframe, date, 12, tab...   

                                 Re

In [7]:
custom_labels = {
    0: 'Basic Data Struct.',
    1: 'PD DataFrame',
    2: 'Environment',
    3: 'Troubleshooting',
    4: 'OO and GUI Prog.',
    5: 'Data Viz. and ML',
    6: 'Web Develop.',
    7: 'PD Missing Data',
}

In [8]:
topic_model.set_topic_labels(custom_labels)

In [9]:
fig1 = topic_model.visualize_barchart(top_n_topics=n_clusters, n_words=10, custom_labels=custom_labels)
fig1.update_layout(font=dict(size=10))
fn = f'figures/topics/python_tbl.pdf'
fig1.write_image(fn)
fig1.show()

In [10]:
umap_embeddings = topic_model.umap_model.transform(np.stack(df_before['embedding'].values))
print(silhouette_score(umap_embeddings, topics))

0.34220666


In [11]:
# topics after
topics, probs = topic_model.fit_transform(df_after['text'], np.stack(df_after['embedding'].values))
print(topic_model.get_topic_info())

   Topic  Count                            Name          CustomName  \
0      0  17436     0_code_class_function_error  Basic Data Struct.   
1      1  12028         1_python_using_use_file        PD DataFrame   
2      2  11489  2_dataframe_column_nan_columns         Environment   
3      3  11257         3_list_output_want_code     Troubleshooting   
4      4  10919       4_file_code_function_like    OO and GUI Prog.   
5      5   8574        5_error_install_file_run    Data Viz. and ML   
6      6   8116         6_plot_model_image_code        Web Develop.   
7      7   6940  7_selenium_page_website_scrape     PD Missing Data   

                                      Representation  \
0  [code, class, function, error, im, button, use...   
1  [python, using, use, file, run, way, im, data,...   
2  [dataframe, column, nan, columns, like, values...   
3  [list, output, want, code, number, values, lik...   
4  [file, code, function, like, im, using, data, ...   
5  [error, install, file

In [12]:
custom_labels = {
    3: 'Basic Data Struct.',
    2: 'PD DataFrame',
    1: 'Environment',
    5: 'Troubleshooting',
    0: 'OO and GUI Prog.',
    6: 'Data Viz. and ML',
    7: 'Web Scraping',
    4: 'File Handling',
}

In [13]:
topic_model.set_topic_labels(custom_labels)

In [14]:
fig2 = topic_model.visualize_barchart(top_n_topics=n_clusters, n_words=10, custom_labels=custom_labels)
fig2.update_layout(font=dict(size=10))
fn = f'figures/topics/python_tal.pdf'
fig2.write_image(fn)
fig2.show()

In [15]:
umap_embeddings = topic_model.umap_model.transform(np.stack(df_after['embedding'].values))
print(silhouette_score(umap_embeddings, topics))

0.33568442
